In [ ]:
import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Conv2D,
    DepthwiseConv2D,
    SeparableConv2D,
    BatchNormalization,
    AveragePooling2D,
    Dropout,
    Flatten,
    Dense,
    Activation
)

from tensorflow.keras.models import Model

In [ ]:
input_layer = Input(
    shape=(64, 64, 32)
)

x = Conv2D(
    16,
    (1, 64),
    padding='same',
    use_bias=False
)(input_layer)

x = BatchNormalization()(x)

x = DepthwiseConv2D(
    (64, 1),
    use_bias=False,
    depth_multiplier=2,
    padding='same'
)(x)

x = BatchNormalization()(x)

x = Activation('elu')(x)

x = AveragePooling2D(
    (2,2)
)(x)

x = Dropout(0.5)(x)

x = SeparableConv2D(
    32,
    (3,3),
    padding='same',
    use_bias=False
)(x)

x = BatchNormalization()(x)

x = Activation('elu')(x)

x = AveragePooling2D(
    (2,2)
)(x)

x = Dropout(0.5)(x)

x = Flatten()(x)

x = Dense(
    64,
    activation='relu'
)(x)

output_layer = Dense(
    2,
    activation='softmax'
)(x)

eegnet_model = Model(
    inputs=input_layer,
    outputs=output_layer
)

eegnet_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

eegnet_model.summary()

In [ ]:
history_eegnet = eegnet_model.fit(
    X_train,
    y_train,
    validation_data=(
        X_test,
        y_test
    ),
    epochs=15,
    batch_size=8
)

In [ ]:
plt.plot(
    history_eegnet.history['accuracy']
)

plt.plot(
    history_eegnet.history['val_accuracy']
)

plt.title(
    "EEGNet Model Accuracy"
)

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.legend([
    'Train',
    'Validation'
])

plt.savefig(
    "/kaggle/working/eegnet_accuracy.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
plt.plot(
    history_eegnet.history['loss']
)

plt.plot(
    history_eegnet.history['val_loss']
)

plt.title(
    "EEGNet Model Loss"
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend([
    'Train',
    'Validation'
])

plt.savefig(
    "/kaggle/working/eegnet_loss.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
sample = X_test[0]

prediction = eegnet_model.predict(
    sample[np.newaxis, ...]
)

predicted_class = np.argmax(
    prediction
)

print(prediction)

print(predicted_class)

In [ ]:
plt.figure(figsize=(8,8))

plt.imshow(
    sample[:, :, 0],
    cmap='inferno'
)

plt.title(
    f"EEGNet Spectrogram | Predicted: {predicted_class}"
)

plt.colorbar()

plt.savefig(
    "/kaggle/working/eegnet_spectrogram.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
grad_model = tf.keras.models.Model(
    [eegnet_model.inputs],
    [
        eegnet_model.get_layer(index=3).output,
        eegnet_model.output
    ]
)

input_image = sample[np.newaxis, ...]

with tf.GradientTape() as tape:

    conv_outputs, predictions = grad_model(
        input_image
    )

    class_idx = tf.argmax(
        predictions[0]
    )

    loss = predictions[:, class_idx]

grads = tape.gradient(
    loss,
    conv_outputs
)

pooled_grads = tf.reduce_mean(
    grads,
    axis=(0,1,2)
)

conv_outputs = conv_outputs[0]

heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]

heatmap = tf.squeeze(heatmap)

heatmap = np.maximum(
    heatmap,
    0
)

heatmap /= np.max(
    heatmap
)

plt.figure(figsize=(8,8))

plt.imshow(
    sample[:, :, 0],
    cmap='gray'
)

plt.imshow(
    heatmap,
    cmap='jet',
    alpha=0.5
)

plt.title(
    "EEGNet GradCAM"
)

plt.colorbar()

plt.savefig(
    "/kaggle/working/eegnet_gradcam.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
real_importance = np.std(
    sample,
    axis=(0,1)
)

real_importance = (
    real_importance -
    np.min(real_importance)
)

real_importance = (
    real_importance /
    np.max(real_importance)
)

fig, ax = plt.subplots(figsize=(8,8))

mne.viz.plot_topomap(
    real_importance,
    info,
    cmap='jet',
    contours=6,
    axes=ax,
    show=False
)

plt.title(
    "EEGNet EEG Topomap"
)

plt.savefig(
    "/kaggle/working/eegnet_topomap.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
predictions = eegnet_model.predict(
    X_test
)

y_pred = np.argmax(
    predictions,
    axis=1
)

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

report = classification_report(
    y_test,
    y_pred,
    zero_division=0
)

print(report)

In [ ]:
X_cnnrnn = X_subj.reshape(
    -1,
    32,
    X_subj.shape[-1]
)

y_cnnrnn = y_bin.reshape(-1)

print(X_cnnrnn.shape)
print(y_cnnrnn.shape)

In [ ]:
X_cnnrnn = np.transpose(
    X_cnnrnn,
    (0, 2, 1)
)

print(X_cnnrnn.shape)

In [ ]:
from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    LSTM,
    Dense,
    Dropout
)

In [ ]:
model = Sequential()

model.add(

    Conv1D(
        64,
        kernel_size=3,
        activation='relu',
        input_shape=(
            X_train.shape[1],
            X_train.shape[2]
        )
    )
)

model.add(
    MaxPooling1D(pool_size=2)
)

model.add(

    Conv1D(
        128,
        kernel_size=3,
        activation='relu'
    )
)

model.add(
    MaxPooling1D(pool_size=2)
)

model.add(

    LSTM(
        64
    )
)

model.add(
    Dropout(0.5)
)

model.add(

    Dense(
        64,
        activation='relu'
    )
)

model.add(

    Dense(
        1,
        activation='sigmoid'
    )
)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()